# 15. Real-cube inference: our trained U-Nets on exoALMA fiducial line images

Jason (2026-09-11 meeting): *"if you want to take the time to do inference on actual ALMA data, this is
probably the data that we're going to start with ... start with the fiducial line images, just the
`.image.fits`, 13CO or 12CO."* MWC 758 was named as a disk people like.

**What this measures, and what it cannot.** A real cube has no ground truth, so nothing here is a PSNR or a
distance-to-truth. It reports what the model *changes*: noise removed off the line, whether flux is conserved
(M0 total), and how far the velocity field (M1) moves. A denoiser that removes 60% of the noise and shifts M1 by
a hundredth of a channel is behaving; one that moves M1 by half a channel or removes flux is inventing
kinematics, which is the failure this project cares most about.

**Preprocessing is training's, exactly** (`src/data/fits_cube_dataset.py`): crop, subtract the cube's own continuum
(mean of the first and last 5 channels), per-channel min-max by that channel's own range, bilinear resize, model,
invert. The one addition is the pixel scale. Training cubes were 600 px at 6.9 mas resized to 256 px, so about
16 mas/px with a 0.14" beam (~9 px). exoALMA is 25 mas/px with a 0.15" beam (~6 px), so the crop is resampled
to ~16 mas/px to hand the model the same beam-in-pixels.

**Caveats to carry into any write-up.** The fiducial images are CLEANed with a uv-taper to a circular 0.15" beam
(exoALMA I, Teague et al. 2025, arXiv:2504.18688), so they carry CLEAN artefacts, not just thermal noise, and
the models were trained on dirty-vs-clean pairs of a different kind. Only the 256px models are run here
(the 320/480/600px arms need a different pixel scale). Off-line rms assumes the first and last 5 channels are line-free.

**Attach:** a Dataset holding `winner_aug_seed43` (required), optionally `winner_p10_seed44`, and optionally the
Output of 05 (for `nb05_winner_{mae,wavelet,starlet,gradient}_ft_seed42`). No line-emission or SG data needed.
Internet ON. The cubes (1.26 GB per line) come from an attached Dataset if one holds them, otherwise from Harvard Dataverse doi:10.7910/DVN/CFHWNH.

## 0. Bootstrap

In [ ]:
import os, sys, subprocess, glob, json, time, shutil, urllib.request

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'alma-validation'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(PKG); sys.path.insert(0, PKG)

    def locate_ckpt(stem):
        hits = [h for ext in ('.pth', '.ckpt', '.pth.tar')
                for h in glob.glob(f'/kaggle/input/**/{stem}{ext}', recursive=True)
                if os.path.isfile(h)]
        return hits[0] if hits else None
    WORK = '/kaggle/working'
else:
    PKG = os.path.abspath('DENOISING_DIFFUSION') if os.path.isdir('DENOISING_DIFFUSION') else os.getcwd()
    os.chdir(PKG); sys.path.insert(0, PKG)
    def locate_ckpt(stem):
        hits = glob.glob(f'models/best_models/**/{stem}.pth', recursive=True)
        return hits[0] if hits else None
    WORK = os.path.join(PKG, 'alma_out')
os.makedirs(WORK, exist_ok=True)
print('cwd:', os.getcwd(), '| ON_KAGGLE:', ON_KAGGLE)

## 0b. Pull latest `src` and `tools` (re-run anytime, no restart needed)

In [ ]:
if ON_KAGGLE:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'], capture_output=True, text=True).stdout)
assert os.path.exists('tools/alma_infer.py'), 'tools/alma_infer.py missing: wrong branch pulled? RULES.md #9'

## 1. Config

In [ ]:
import torch

# (disk, line) pairs to run. 13CO is the canonical line, 12CO is fine, CS is too dim (Jason).
# Each cube is 1.26 GB. Filenames are exoALMA's: <disk>_<line>_fiducial.image.fits
CUBES = [('MWC_758', '12CO'), ('MWC_758', '13CO')]

# label -> checkpoint stem. aug43 is the reference model; the rest are used if found.
CKPT_STEMS = {
    'aug43':       'winner_aug_seed43',
    'p10_44':      'winner_p10_seed44',
    'mae_ft':      'nb05_winner_mae_ft_seed42',
    'wavelet_ft':  'nb05_winner_wavelet_ft_seed42',
    'starlet_ft':  'nb05_winner_starlet_ft_seed42',
    'gradient_ft': 'nb05_winner_gradient_ft_seed42',
}
FOV_ARCSEC = 8.0     # central field kept; MWC 758's CO disk is ~4" in radius
PIX_MAS = 16.0       # model pixel scale; matches training (4.13" over 256 px)

CKPTS = {}
for label, stem in CKPT_STEMS.items():
    p = locate_ckpt(stem)
    print(f'{label:12s} {stem:34s}', p or 'NOT FOUND (skipped)')
    if p: CKPTS[label] = p
assert 'aug43' in CKPTS, 'winner_aug_seed43 is the reference model and must be attached'
print('\ndevice:', 'cuda ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Get the exoALMA cubes

Two routes, tried in this order: (1) cubes already attached as a Kaggle Dataset (any folder under `/kaggle/input`
holding `<disk>_<line>_fiducial.image.fits`), (2) fetched from Harvard Dataverse. Attaching them skips the
1.26 GB download per cube on every session, but a home connection makes uploading them slow, so route 2 is
the sensible default unless the Dataset already exists.

In [ ]:
DATAVERSE, DOI = 'https://dataverse.harvard.edu', 'doi:10.7910/DVN/CFHWNH'
CUBE_DIR = os.path.join(WORK, 'exoalma'); os.makedirs(CUBE_DIR, exist_ok=True)
_files = None
CUBE_PATH = {}
for disk, line in CUBES:
    name = f'{disk}_{line}_fiducial.image.fits'
    attached = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isfile(h)]
    if attached:
        CUBE_PATH[(disk, line)] = attached[0]
        print(f'{name}: attached Dataset, {os.path.getsize(attached[0]) / 1e6:.0f} MB -> {attached[0]}')
        continue
    if _files is None:
        with urllib.request.urlopen(f'{DATAVERSE}/api/datasets/:persistentId/?persistentId={DOI}', timeout=60) as r:
            _files = {f['dataFile']['filename']: f['dataFile'] for f in json.load(r)['data']['latestVersion']['files']}
    assert name in _files, f'{name} not in the release; e.g. {sorted(_files)[:3]}'
    dst = os.path.join(CUBE_DIR, name); want = _files[name]['filesize']
    if not (os.path.exists(dst) and os.path.getsize(dst) == want):
        t0 = time.time()
        subprocess.run(['curl', '-sL', '--retry', '5', '--retry-delay', '10', '-C', '-', '-o', dst,
                        f"{DATAVERSE}/api/access/datafile/{_files[name]['id']}"], check=True)
        print(f'{name}: downloaded {os.path.getsize(dst) / 1e6:.0f} MB in {time.time() - t0:.0f} s')
    assert os.path.getsize(dst) == want, f'{name}: {os.path.getsize(dst)} bytes, expected {want} (truncated download)'
    CUBE_PATH[(disk, line)] = dst
print('cubes ready:', [os.path.basename(p) for p in CUBE_PATH.values()])

## 3. Inference

One `alma_infer.py` call per cube, all checkpoints together so they share the same raw moment maps.
It writes `report.txt`, `moments.png`, `channel.png` and one `<label>_denoised.fits` per checkpoint.

In [ ]:
RUNS = {}
for (disk, line), cube in CUBE_PATH.items():
    tag = f'{disk}_{line}'
    out = os.path.join(WORK, f'nb15_{tag}'); os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, 'tools/alma_infer.py', '--cube', cube, '--out', out,
           '--fov', str(FOV_ARCSEC), '--pix-mas', str(PIX_MAS),
           '--device', 'cuda' if torch.cuda.is_available() else 'cpu', '--batch', '8',
           '--ckpt'] + [f'{l}={p}' for l, p in CKPTS.items()]
    print(f'\n{"=" * 78}\n=== {tag}\n{"=" * 78}', flush=True)
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f'[{tag}: {time.time() - t0:.0f} s]')
    # flat, prefixed copies at the top of /kaggle/working so collect_outputs and the Output tab find them
    for f in ('report.txt', 'moments.png', 'channel.png'):
        shutil.copy2(os.path.join(out, f), os.path.join(WORK, f'nb15_{tag}_{f}'))
    RUNS[tag] = out

## 4. What the models did

In [ ]:
from IPython.display import Image, display
for tag, out in RUNS.items():
    print(f'\n##### {tag}')
    print(open(os.path.join(out, 'report.txt')).read())
    display(Image(os.path.join(out, 'moments.png')))
    display(Image(os.path.join(out, 'channel.png')))

## 5. Collect outputs

In [ ]:
from src.evaluation.collect_outputs import collect_outputs

_run_dir = collect_outputs(
    '15-alma-real-cube-inference',
    ['nb15_*_report.txt', 'nb15_*_moments.png', 'nb15_*_channel.png'],
    extra={'cubes': [f'{d}_{l}' for d, l in CUBES], 'checkpoints': sorted(CKPTS),
           'fov_arcsec': FOV_ARCSEC, 'pix_mas': PIX_MAS},
)
print('\nAnything listed as NOT FOUND above did not get written this run.')